# MobileRec: NeuMF Neural Recommendation Model

This study uses a NeuMF architecture adapted for ranking with Bayesian Personalized Ranking (BPR) loss. User and app embeddings are learned in two branches. The Generalized Matrix Factorization (GMF) branch combines embeddings through element-wise multiplication, while the multilayer perceptron (MLP) branch learns nonlinear relationships through several neural layers. Their outputs are combined into a single score for each user–app pair.

NeuMF provides a different neural approach from the autoencoders and LightGCN: it scores individual user–app pairs rather than taking a complete history vector as input or propagating information through a graph.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import copy
import math
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.sparse import load_npz

In [2]:
run_name = "all_users"
seed = 42
top_k = 10
batch_size = 16384
eval_batch = 64
max_epochs = 20
patience = 3
min_improvement = 0.00001
lr = 0.001
reg = 0.00001

All users are included, with `seed = 42` and ten recommendations per user. `batch_size` controls the number of sampled positive and negative pairs in each training batch, while `eval_batch` specifies how many users are processed together during evaluation.

The epoch limit, early-stopping rule, learning rate, and regularization are determined before testing. The best checkpoint is selected using validation NDCG@10.

In [3]:
split_dir = Path("artifacts") / "split_all_users"
base_dir = Path("artifacts") / "baselines_all_users"
autoencoders_dir = Path("artifacts") / "autoencoders_all_users"
results_dir = Path("artifacts") / "neumf_all_users"

train_path = split_dir / "train_all_users.parquet"
valid_path = split_dir / "validation_all_users.parquet"
test_path = split_dir / "test_all_users.parquet"
user_map_path = split_dir / "users_map_all_users.parquet"
item_map_path = split_dir / "items_map_all_users.parquet"
matrix_path = base_dir / f"user_item_matrix_{run_name}_seed_{seed}.npz"
all_results_path = autoencoders_dir / f"all_results_{run_name}_seed_{seed}.csv"

results_dir.mkdir(parents=True, exist_ok=True)

for path in [train_path, valid_path, test_path, user_map_path, item_map_path, matrix_path, all_results_path]:
    if not path.exists():
        raise FileNotFoundError(f"File does not exist: {path}")

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

set_seed(seed)

print("Uredjaj:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Uredjaj: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


Paths to the chronological splits, mappings, sparse training matrix, and previous results table are configured. The model is trained on the GPU.

In [5]:
user_map = pd.read_parquet(user_map_path).sort_values("user_idx").reset_index(drop=True)
item_map = pd.read_parquet(item_map_path).sort_values("item_idx").reset_index(drop=True)
user_item_matrix = load_npz(matrix_path).tocsr()

num_users, num_items = user_item_matrix.shape
users, apps = user_item_matrix.nonzero()
num_pairs = len(users)

assert num_users == len(user_map)
assert num_items == len(item_map)

print("Users:", num_users)
print("Candidate applications:", num_items)
print("Training user–application pairs:", num_pairs)

Users: 700111
Candidate applications: 10172
Training user–application pairs: 17748311


In [6]:
def load_targets(path):
    data = pd.read_parquet(path, columns=["uid", "app_package"])
    data = data.merge(user_map, on="uid", how="left")
    data = data.merge(item_map, on="app_package", how="left")
    targets = data.dropna(subset=["item_idx"]).copy()
    targets["user_idx"] = targets["user_idx"].astype(int)
    targets["item_idx"] = targets["item_idx"].astype(int)
    return data, targets.sort_values("user_idx").reset_index(drop=True)

valid_data, valid_targets = load_targets(valid_path)
test_data, test_targets = load_targets(test_path)

valid_coverage = len(valid_targets) / len(valid_data) * 100
test_coverage = len(test_targets) / len(test_data) * 100

print("Validation targets:", len(valid_targets))
print("Validation coverage:", round(valid_coverage, 4), "%")
print("Test targets:", len(test_targets))
print("Test coverage:", round(test_coverage, 4), "%")

Validation targets: 700108
Validation coverage: 99.9996 %
Test targets: 700096
Test coverage: 99.9979 %


NeuMF uses the same sparse training matrix and numeric mappings as the previous personalized models. The three validation and 15 test targets whose app is absent from training are excluded from standard ranking metrics, and candidate-set coverage is reported separately.

In [7]:
class NeuMF(nn.Module):
    def __init__(self, num_users, num_items, dim, mlp_layers):
        super().__init__()
        self.gmf_user = nn.Embedding(num_users, dim)
        self.gmf_app = nn.Embedding(num_items, dim)
        self.mlp_user = nn.Embedding(num_users, dim)
        self.mlp_app = nn.Embedding(num_items, dim)
        layers = []
        input_dim = dim * 2

        for output_dim in mlp_layers:
            layers.append(nn.Linear(input_dim, output_dim))
            layers.append(nn.ReLU())
            input_dim = output_dim

        self.mlp = nn.Sequential(*layers)
        self.output = nn.Linear(dim + input_dim, 1)
        self.reset_parameters()

    def reset_parameters(self):
        for layer in [self.gmf_user, self.gmf_app, self.mlp_user, self.mlp_app]:
            nn.init.normal_(layer.weight, std=0.01)
            
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

        nn.init.xavier_uniform_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def score_pairs(self, users, apps):
        gmf = self.gmf_user(users) * self.gmf_app(apps)
        mlp_input = torch.cat([self.mlp_user(users), self.mlp_app(apps)], dim=1)
        mlp = self.mlp(mlp_input)
        return self.output(torch.cat([gmf, mlp], dim=1)).squeeze(1)

    def score_all_apps(self, users):
        size = len(users)
        gmf = self.gmf_user(users).unsqueeze(1) * self.gmf_app.weight.unsqueeze(0)
        user_emb = self.mlp_user(users).unsqueeze(1).expand(size, num_items, -1)
        app_emb = self.mlp_app.weight.unsqueeze(0).expand(size, num_items, -1)
        mlp_input = torch.cat([user_emb, app_emb], dim=2)
        mlp = self.mlp(mlp_input.reshape(size * num_items, -1))
        mlp = mlp.reshape(size, num_items, -1)
        return self.output(torch.cat([gmf, mlp], dim=2)).squeeze(2)

NeuMF has GMF and MLP branches. GMF multiplies the user and app embeddings element by element. MLP concatenates separate user and app embeddings and processes them through the configured layers. A final linear layer converts the combined branch outputs into a single score.

During training, `score_pairs` scores sampled positive and negative pairs. During evaluation, `score_all_apps` scores every candidate app for a batch of users.

In [8]:
user_apps = [set(user_item_matrix.getrow(i).indices) for i in range(num_users)]

def sample_bpr(rng):
    pair_pos = rng.integers(num_pairs, size=batch_size)
    batch_users = users[pair_pos]
    positive_apps = apps[pair_pos]
    negative_apps = np.empty(batch_size, dtype=np.int64)

    for i, user in enumerate(batch_users):
        app = rng.integers(num_items)
        while app in user_apps[user]:
            app = rng.integers(num_items)
        negative_apps[i] = app

    return (torch.from_numpy(batch_users).to(device),
            torch.from_numpy(positive_apps).to(device),
            torch.from_numpy(negative_apps).to(device))

In [9]:
def train_epoch(model, optimizer, rng):
    model.train()
    loss_sum = 0
    batch_num = math.ceil(num_pairs / batch_size)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    start_time = perf_counter()

    for _ in range(batch_num):
        batch_users, positive_apps, negative_apps = sample_bpr(rng)
        optimizer.zero_grad()
        positive_score = model.score_pairs(batch_users, positive_apps)
        negative_score = model.score_pairs(batch_users, negative_apps)
        bpr_loss = -F.logsigmoid(positive_score - negative_score).mean()
        reg_loss = 0.5 * sum(parameter.pow(2).mean() for parameter in model.parameters())
        loss = bpr_loss + reg * reg_loss
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()

    peak_memory = None

    if device.type == "cuda":
        peak_memory = torch.cuda.max_memory_allocated() / 1024 / 1024

    return loss_sum / batch_num, perf_counter() - start_time, peak_memory


Each training batch samples observed positive pairs from the training matrix and negative apps absent from the selected users’ training histories. Here, “negative” means unobserved in training, not disliked by the user. BPR loss encourages the positive pair to receive a higher score than the negative pair. An additional `reg_loss` term penalizes excessive parameter magnitudes.

In [10]:
def eval_top_k(model, targets):
    model.eval()
    hit_sum = 0
    ndcg_sum = 0

    if device.type == "cuda":
        torch.cuda.synchronize()
    start_time = perf_counter()

    with torch.inference_mode():
        for start in range(0, len(targets), eval_batch):
            batch = targets.iloc[start:start + eval_batch]
            batch_users = torch.tensor(batch["user_idx"].to_numpy(), device=device)
            target_apps = torch.tensor(batch["item_idx"].to_numpy(), device=device)
            scores = model.score_all_apps(batch_users)
            rec = torch.argsort(scores, dim=1, descending=True, stable=True)[:, :top_k]
            matches = rec.eq(target_apps.unsqueeze(1))
            hits = matches.any(dim=1)
            ranks = matches.to(torch.int64).argmax(dim=1)
            hit_sum += hits.sum().item()
            ndcg_sum += (hits / torch.log2(ranks + 2)).sum().item()

    if device.type == "cuda":
        torch.cuda.synchronize()

    return {"hit_recall_at_10": hit_sum / len(targets),
            "ndcg_at_10": ndcg_sum / len(targets),
            "evaluation_seconds": perf_counter() - start_time}

The `eval_top_k` function scores all candidate apps, selects the top ten, and computes Hit/Recall@10 and NDCG@10 using the same protocol as the other models.

In [11]:
def run_neumf(config):
    set_seed(seed)
    rng = np.random.default_rng(seed)
    model = NeuMF(num_users, num_items, config["dim"], config["mlp_layers"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_ndcg = -1
    best_epoch = 0
    no_change = 0
    best_state = None
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss, epoch_time, peak_memory = train_epoch(model, optimizer, rng)
        valid_eval = eval_top_k(model, valid_targets)
        history.append({"epoch": epoch,
                        "train_loss": train_loss,
                        "validation_hit_recall_10": valid_eval["hit_recall_at_10"],
                        "validation_ndcg_10": valid_eval["ndcg_at_10"],
                        "training_seconds": epoch_time,
                        "peak_gpu_memory_mib": peak_memory})
        print(f"{config['name']} | epoch {epoch:02d} | loss: {train_loss:.4f} | NDCG@10: {valid_eval['ndcg_at_10']:.6f}")

        if valid_eval["ndcg_at_10"] > best_ndcg + min_improvement:
            best_ndcg = valid_eval["ndcg_at_10"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            no_change = 0
        else:
            no_change += 1

        if no_change >= patience:
            break

    history = pd.DataFrame(history)
    best_row = history.loc[history["epoch"] == best_epoch].iloc[0]

    result = {"name": config["name"],
              "dim": config["dim"],
              "mlp_layers": config["mlp_layers"],
              "best_epoch": best_epoch,
              "validation_hit_recall_10": best_row["validation_hit_recall_10"],
              "validation_ndcg_10": best_ndcg,
              "training_seconds": history["training_seconds"].sum(),
              "peak_gpu_memory_mib": history["peak_gpu_memory_mib"].max()}

    return best_state, history, result

The `run_neumf` function trains one configuration and checks validation NDCG@10 after each epoch. The best checkpoint is retained, and training stops after three consecutive epochs without sufficient improvement.

The first tuning round varies user and app embedding dimensions and MLP layer sizes. Three architectures are compared while the other hyperparameters remain fixed. Selection is based on validation NDCG@10 rather than training loss.

In [12]:
neumf_options = [
    {"name": "NeuMF_1", "dim": 16, "mlp_layers": [32, 16, 8]},
    {"name": "NeuMF_2", "dim": 32, "mlp_layers": [64, 32, 16]},
    {"name": "NeuMF_3", "dim": 64, "mlp_layers": [128, 64, 32]}
]

neumf_rows = []
best_neumf_state = None
best_neumf_history = None
best_neumf_result = None

for config in neumf_options:
    state, history, result = run_neumf(config)
    neumf_rows.append(result)

    if best_neumf_result is None or result["validation_ndcg_10"] > best_neumf_result["validation_ndcg_10"]:
        best_neumf_state = state
        best_neumf_history = history
        best_neumf_result = result

    if device.type == "cuda":
        torch.cuda.empty_cache()

neumf_tuning = pd.DataFrame(neumf_rows)
neumf_tuning = neumf_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
neumf_tuning.to_csv(results_dir / f"neumf_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Izabrani NeuMF:", best_neumf_result["name"])
neumf_tuning.round(6)

NeuMF_1 | epoch 01 | loss: 0.4958 | NDCG@10: 0.005950
NeuMF_1 | epoch 02 | loss: 0.4712 | NDCG@10: 0.005204
NeuMF_1 | epoch 03 | loss: 0.4535 | NDCG@10: 0.003555
NeuMF_1 | epoch 04 | loss: 0.4402 | NDCG@10: 0.002744
NeuMF_2 | epoch 01 | loss: 0.4910 | NDCG@10: 0.005866
NeuMF_2 | epoch 02 | loss: 0.4591 | NDCG@10: 0.004120
NeuMF_2 | epoch 03 | loss: 0.4270 | NDCG@10: 0.002004
NeuMF_2 | epoch 04 | loss: 0.4035 | NDCG@10: 0.001668
NeuMF_3 | epoch 01 | loss: 0.4875 | NDCG@10: 0.005914
NeuMF_3 | epoch 02 | loss: 0.4396 | NDCG@10: 0.002940
NeuMF_3 | epoch 03 | loss: 0.3883 | NDCG@10: 0.001259
NeuMF_3 | epoch 04 | loss: 0.3543 | NDCG@10: 0.000912
Izabrani NeuMF: NeuMF_1


,name,dim,mlp_layers,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib
0,NeuMF_1,16,"[32, 16, 8]",1,0.011621,0.005950,848.201562,597.328125
1,NeuMF_3,64,"[128, 64, 32]",1,0.011620,0.005914,1001.599789,2600.153320
2,NeuMF_2,32,"[64, 32, 16]",1,0.011525,0.005866,967.702432,1271.122070


The smallest architecture, `NeuMF_1`, performs best in the first round, with an embedding dimension of 16 and MLP layers `[32, 16, 8]`. It achieves validation Hit/Recall@10 of 1.1621% and NDCG@10 of 0.5950%. Larger architectures do not improve performance.

For all three configurations, validation performance declines as training loss decreases, suggesting possible overfitting. The smallest architecture is retained, and lower learning rates are tested next.

In [13]:
old_lr = lr
old_reg = reg

neumf_extra_options = [
    {"name": "NeuMF_4", "dim": 16, "mlp_layers": [32, 16, 8], "lr": 0.0005, "reg": 0.00001},
    {"name": "NeuMF_5", "dim": 16, "mlp_layers": [32, 16, 8], "lr": 0.00025, "reg": 0.00001}
]

for config in neumf_extra_options:
    lr = config["lr"]
    reg = config["reg"]
    state, history, result = run_neumf(config)
    result["lr"] = lr
    result["reg"] = reg
    neumf_rows.append(result)

    if result["validation_ndcg_10"] > best_neumf_result["validation_ndcg_10"]:
        best_neumf_state = state
        best_neumf_history = history
        best_neumf_result = result

    if device.type == "cuda":
        torch.cuda.empty_cache()

lr = old_lr
reg = old_reg

neumf_tuning = pd.DataFrame(neumf_rows)
neumf_tuning = neumf_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
neumf_tuning.to_csv(results_dir / f"neumf_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Konacno izabrani NeuMF:", best_neumf_result["name"])
neumf_tuning.round(6)

NeuMF_4 | epoch 01 | loss: 0.5043 | NDCG@10: 0.006018
NeuMF_4 | epoch 02 | loss: 0.4808 | NDCG@10: 0.005933
NeuMF_4 | epoch 03 | loss: 0.4719 | NDCG@10: 0.005706
NeuMF_4 | epoch 04 | loss: 0.4597 | NDCG@10: 0.005420
NeuMF_5 | epoch 01 | loss: 0.5192 | NDCG@10: 0.005963
NeuMF_5 | epoch 02 | loss: 0.4839 | NDCG@10: 0.006174
NeuMF_5 | epoch 03 | loss: 0.4811 | NDCG@10: 0.006161
NeuMF_5 | epoch 04 | loss: 0.4766 | NDCG@10: 0.005887
NeuMF_5 | epoch 05 | loss: 0.4714 | NDCG@10: 0.005987
Konacno izabrani NeuMF: NeuMF_5


,name,dim,mlp_layers,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib,lr,reg
0,NeuMF_5,16,"[32, 16, 8]",2,0.012470,0.006174,1054.930668,684.041992,0.00025,0.00001
1,NeuMF_4,16,"[32, 16, 8]",1,0.011620,0.006018,843.510747,1032.480957,0.00050,0.00001
2,NeuMF_1,16,"[32, 16, 8]",1,0.011621,0.005950,848.201562,597.328125,NaN,NaN
3,NeuMF_3,64,"[128, 64, 32]",1,0.011620,0.005914,1001.599789,2600.153320,NaN,NaN
4,NeuMF_2,32,"[64, 32, 16]",1,0.011525,0.005866,967.702432,1271.122070,NaN,NaN


Lowering the learning rate improves performance. `NeuMF_5`, with a learning rate of 0.00025, achieves validation NDCG@10 of 0.006174 in the second epoch. Subsequent experiments test stronger regularization and still lower learning rates.

In [14]:
old_lr = lr
old_reg = reg

config = {"name": "NeuMF_6", "dim": 16, "mlp_layers": [32, 16, 8],
          "lr": 0.0005, "reg": 0.00005}

lr = config["lr"]
reg = config["reg"]
state, history, result = run_neumf(config)
result["lr"] = lr
result["reg"] = reg
neumf_rows.append(result)

if result["validation_ndcg_10"] > best_neumf_result["validation_ndcg_10"]:
    best_neumf_state = state
    best_neumf_history = history
    best_neumf_result = result

lr = old_lr
reg = old_reg

neumf_tuning = pd.DataFrame(neumf_rows)
neumf_tuning = neumf_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
neumf_tuning.to_csv(results_dir / f"neumf_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Konacno izabrani NeuMF:", best_neumf_result["name"])
neumf_tuning.round(6)

NeuMF_6 | epoch 01 | loss: 0.5043 | NDCG@10: 0.006029
NeuMF_6 | epoch 02 | loss: 0.4810 | NDCG@10: 0.005897
NeuMF_6 | epoch 03 | loss: 0.4723 | NDCG@10: 0.005684
NeuMF_6 | epoch 04 | loss: 0.4602 | NDCG@10: 0.005473
Konacno izabrani NeuMF: NeuMF_5


,name,dim,mlp_layers,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib,lr,reg
0,NeuMF_5,16,"[32, 16, 8]",2,0.012470,0.006174,1054.930668,684.041992,0.00025,0.00001
1,NeuMF_6,16,"[32, 16, 8]",1,0.011671,0.006029,840.090682,684.041992,0.00050,0.00005
2,NeuMF_4,16,"[32, 16, 8]",1,0.011620,0.006018,843.510747,1032.480957,0.00050,0.00001
3,NeuMF_1,16,"[32, 16, 8]",1,0.011621,0.005950,848.201562,597.328125,NaN,NaN
4,NeuMF_3,64,"[128, 64, 32]",1,0.011620,0.005914,1001.599789,2600.153320,NaN,NaN
5,NeuMF_2,32,"[64, 32, 16]",1,0.011525,0.005866,967.702432,1271.122070,NaN,NaN


In [15]:
old_lr = lr
old_reg = reg

config = {"name": "NeuMF_7", "dim": 16, "mlp_layers": [32, 16, 8],
          "lr": 0.0001, "reg": 0.00001}

lr = config["lr"]
reg = config["reg"]
state, history, result = run_neumf(config)
result["lr"] = lr
result["reg"] = reg
neumf_rows.append(result)

if result["validation_ndcg_10"] > best_neumf_result["validation_ndcg_10"]:
    best_neumf_state = state
    best_neumf_history = history
    best_neumf_result = result

lr = old_lr
reg = old_reg

neumf_tuning = pd.DataFrame(neumf_rows)
neumf_tuning = neumf_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
neumf_tuning.to_csv(results_dir / f"neumf_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Konacno izabrani NeuMF:", best_neumf_result["name"])
neumf_tuning.round(6)

NeuMF_7 | epoch 01 | loss: 0.5627 | NDCG@10: 0.005934
NeuMF_7 | epoch 02 | loss: 0.4853 | NDCG@10: 0.006059
NeuMF_7 | epoch 03 | loss: 0.4836 | NDCG@10: 0.006337
NeuMF_7 | epoch 04 | loss: 0.4827 | NDCG@10: 0.005817
NeuMF_7 | epoch 05 | loss: 0.4819 | NDCG@10: 0.006099
NeuMF_7 | epoch 06 | loss: 0.4810 | NDCG@10: 0.006302
Konacno izabrani NeuMF: NeuMF_7


,name,dim,mlp_layers,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib,lr,reg
0,NeuMF_7,16,"[32, 16, 8]",3,0.012295,0.006337,1262.845831,770.755859,0.00010,0.00001
1,NeuMF_5,16,"[32, 16, 8]",2,0.012470,0.006174,1054.930668,684.041992,0.00025,0.00001
2,NeuMF_6,16,"[32, 16, 8]",1,0.011671,0.006029,840.090682,684.041992,0.00050,0.00005
3,NeuMF_4,16,"[32, 16, 8]",1,0.011620,0.006018,843.510747,1032.480957,0.00050,0.00001
4,NeuMF_1,16,"[32, 16, 8]",1,0.011621,0.005950,848.201562,597.328125,NaN,NaN
5,NeuMF_3,64,"[128, 64, 32]",1,0.011620,0.005914,1001.599789,2600.153320,NaN,NaN
6,NeuMF_2,32,"[64, 32, 16]",1,0.011525,0.005866,967.702432,1271.122070,NaN,NaN


In [16]:
old_lr = lr
old_reg = reg

new_options = [
    {"name": "NeuMF_8", "dim": 16, "mlp_layers": [32, 16, 8],
     "lr": 0.00005, "reg": 0.00001},
    {"name": "NeuMF_9", "dim": 32, "mlp_layers": [64, 32, 16],
     "lr": 0.0001, "reg": 0.00001}
]

for config in new_options:
    lr = config["lr"]
    reg = config["reg"]
    state, history, result = run_neumf(config)
    result["lr"] = lr
    result["reg"] = reg
    neumf_rows.append(result)

    if result["validation_ndcg_10"] > best_neumf_result["validation_ndcg_10"]:
        best_neumf_state = state
        best_neumf_history = history
        best_neumf_result = result

    if device.type == "cuda":
        torch.cuda.empty_cache()

lr = old_lr
reg = old_reg

neumf_tuning = pd.DataFrame(neumf_rows)
neumf_tuning = neumf_tuning.sort_values("validation_ndcg_10", ascending=False).reset_index(drop=True)
neumf_tuning.to_csv(results_dir / f"neumf_tuning_{run_name}_seed_{seed}.csv", index=False)

print("Konacno izabrani NeuMF:", best_neumf_result["name"])
neumf_tuning.round(6)

NeuMF_8 | epoch 01 | loss: 0.6240 | NDCG@10: 0.004718
NeuMF_8 | epoch 02 | loss: 0.4996 | NDCG@10: 0.005943
NeuMF_8 | epoch 03 | loss: 0.4860 | NDCG@10: 0.006037
NeuMF_8 | epoch 04 | loss: 0.4840 | NDCG@10: 0.005845
NeuMF_8 | epoch 05 | loss: 0.4830 | NDCG@10: 0.005811
NeuMF_8 | epoch 06 | loss: 0.4824 | NDCG@10: 0.006405
NeuMF_8 | epoch 07 | loss: 0.4818 | NDCG@10: 0.006319
NeuMF_8 | epoch 08 | loss: 0.4815 | NDCG@10: 0.006360
NeuMF_8 | epoch 09 | loss: 0.4810 | NDCG@10: 0.006403
NeuMF_9 | epoch 01 | loss: 0.5309 | NDCG@10: 0.006118
NeuMF_9 | epoch 02 | loss: 0.4839 | NDCG@10: 0.006355
NeuMF_9 | epoch 03 | loss: 0.4816 | NDCG@10: 0.006158
NeuMF_9 | epoch 04 | loss: 0.4788 | NDCG@10: 0.005807
NeuMF_9 | epoch 05 | loss: 0.4760 | NDCG@10: 0.005826
Konacno izabrani NeuMF: NeuMF_8


,name,dim,mlp_layers,best_epoch,validation_hit_recall_10,validation_ndcg_10,training_seconds,peak_gpu_memory_mib,lr,reg
0,NeuMF_8,16,"[32, 16, 8]",6,0.012481,0.006405,1899.168231,684.041992,0.00005,0.00001
1,NeuMF_9,32,"[64, 32, 16]",2,0.012481,0.006355,1215.438124,1271.122070,0.00010,0.00001
2,NeuMF_7,16,"[32, 16, 8]",3,0.012295,0.006337,1262.845831,770.755859,0.00010,0.00001
3,NeuMF_5,16,"[32, 16, 8]",2,0.012470,0.006174,1054.930668,684.041992,0.00025,0.00001
4,NeuMF_6,16,"[32, 16, 8]",1,0.011671,0.006029,840.090682,684.041992,0.00050,0.00005
5,NeuMF_4,16,"[32, 16, 8]",1,0.011620,0.006018,843.510747,1032.480957,0.00050,0.00001
6,NeuMF_1,16,"[32, 16, 8]",1,0.011621,0.005950,848.201562,597.328125,NaN,NaN
7,NeuMF_3,64,"[128, 64, 32]",1,0.011620,0.005914,1001.599789,2600.153320,NaN,NaN
8,NeuMF_2,32,"[64, 32, 16]",1,0.011525,0.005866,967.702432,1271.122070,NaN,NaN


Stronger regularization in `NeuMF_6` does not improve performance. A learning rate of 0.0001 in `NeuMF_7` raises NDCG@10 to 0.006337. The best configuration is `NeuMF_8`, with an embedding dimension of 16, MLP layers `[32, 16, 8]`, a learning rate of 0.00005, and regularization of 0.00001. In the sixth epoch, it achieves validation Hit/Recall@10 of 1.2481% and NDCG@10 of 0.6405%.

The larger `NeuMF_9`, with an embedding dimension of 32 and learning rate of 0.0001, achieves a slightly lower NDCG@10 of 0.006355. Model capacity is therefore not increased further.

In [17]:
neumf_model = NeuMF(num_users, num_items, best_neumf_result["dim"],
                    best_neumf_result["mlp_layers"]).to(device)
neumf_model.load_state_dict(best_neumf_state)
neumf_test = eval_top_k(neumf_model, test_targets)

neumf_result = pd.DataFrame([{
    "model": "NeuMF",
    "users": num_users,
    "seed": seed,
    "validation_Hit@10": best_neumf_result["validation_hit_recall_10"],
    "validation_Recall@10": best_neumf_result["validation_hit_recall_10"],
    "validation_NDCG@10": best_neumf_result["validation_ndcg_10"],
    "test_Hit@10": neumf_test["hit_recall_at_10"],
    "test_Recall@10": neumf_test["hit_recall_at_10"],
    "test_NDCG@10": neumf_test["ndcg_at_10"],
    "test_coverage": test_coverage,
    "model_time_seconds": best_neumf_result["training_seconds"],
    "notes": f"dim={best_neumf_result['dim']}, MLP={best_neumf_result['mlp_layers']}, BPR"
}])

best_neumf_history.to_csv(results_dir / f"neumf_history_{run_name}_seed_{seed}.csv", index=False)
torch.save({"state_dict": neumf_model.state_dict(), "result": best_neumf_result},
           results_dir / f"neumf_{run_name}_seed_{seed}.pt")

print("NeuMF test:", neumf_test)

NeuMF test: {'hit_recall_at_10': 0.015043651156412834, 'ndcg_at_10': 0.007753549840227035, 'evaluation_seconds': 81.26787999999942}


The selected `NeuMF_8` achieves test Hit/Recall@10 of 1.5044% and NDCG@10 of 0.7754%. Its test performance is almost identical to Popularity, although all NeuMF hyperparameters are selected exclusively using the validation set.

In [18]:
old_results = pd.read_csv(all_results_path)
all_results = pd.concat([old_results, neumf_result], ignore_index=True)
all_results = all_results.sort_values("test_NDCG@10", ascending=False).reset_index(drop=True)

all_results.to_csv(results_dir / f"all_results_{run_name}_seed_{seed}.csv", index=False)

show_cols = ["model", "validation_Hit@10", "validation_NDCG@10",
             "test_Hit@10", "test_NDCG@10", "model_time_seconds", "notes"]

all_results[show_cols].round(6)

,model,validation_Hit@10,validation_NDCG@10,test_Hit@10,test_NDCG@10,model_time_seconds,notes
0,ItemKNN,0.013441,0.006680,0.016872,0.008256,36.727301,Binarni ItemKNN sa k = 200
1,Popularity,0.012481,0.006411,0.015051,0.007762,0.227138,Globalna popularnost iz train skupa
2,NeuMF,0.012481,0.006405,0.015044,0.007754,1899.168231,"dim=16, MLP=[32, 16, 8], BPR"
3,MultiVAE,0.012002,0.005918,0.013860,0.006912,583.880002,"hidden=256, latent=128, dropout=0.5, max_kl=0.3"
4,AutoRec,0.009728,0.004480,0.010557,0.004554,533.888907,"latent=128, dropout=0.4, lr=0.0001"


Among the models evaluated so far, ItemKNN remains the strongest with test NDCG@10 of 0.8256%. Popularity achieves 0.7762%, and NeuMF achieves 0.7754%, leaving a very small difference between the latter two. NeuMF outperforms both autoencoders but does not exceed either baseline.

These results show that a more complex personalized neural architecture does not automatically improve on popularity and item similarity for this dataset.